# P4 — The walk operator **is** attention

The central idea. A Transformer updates token `j` by a weighted sum `h_j = Σ_i α_{ji} v_i` with **learned**
weights over **all pairs**. Aggregating over length-`g` walks has the *same shape* — and the path algebra
supplies a **sparse multi-hop support**: `i` attends to `j` only when a length-`g` walk connects them.

**5 figures:** attention as arrow-thickness · the sparse masks · the mask as a matrix · fixed-vs-learned
weights · the learned attention matrix.

In [ ]:
import torch, numpy as np
from oversquash import viz
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
from oversquash.attention import WalkAttention
from torch_geometric.utils import softmax as pyg_softmax
data = make_bottleneck_graph(5, 4, 3, torch.Generator().manual_seed(0))
ei, N = data.edge_index.numpy(), data.num_nodes
raw, _ = walk_operators(ei, N, max_length=4)
t = int(data.root_mask.nonzero()[0])

## Figure 1 — attention is just a weight on each arrow

Three sources point at a target. **Fixed** weights (left) make every arrow equally thick — the model can't
tell sources apart. **Learned** attention (right) makes the relevant arrow thick — it *focuses*. That is the
entire difference.

In [ ]:
ea = [(0,3),(1,3),(2,3)]; pa = {0:(0,1),1:(0,0),2:(0,-1),3:(1.6,0)}
viz.explain_attention_on_graph(ea, pa,
    weight_sets=[{(0,3):1,(1,3):1,(2,3):1}, {(0,3):0.1,(1,3):3.0,(2,3):0.1}],
    n_nodes=4, src_nodes=[0,1,2], dst=3,
    titles=['fixed: all equal (can\'t select)', 'learned: focused (selects source 1)'],
    suptitle='attention = arrow thickness')

## Figure 2 — the attention support is sparse (and the algebra picks it)

Green = a pair `(i,j)` joined by a length-`g` walk. Standard attention would fill the whole square; the path
algebra keeps only ~2% at the deepest range. Global reach, tiny cost.

In [ ]:
viz.plot_mask_grid(raw, N, title='walk-reachability masks = the attention support')

## Figure 3 — fixed vs learned weights (numbers)

Over the **same** reachable sources into the target: `walkraw` uses fixed weights (raw path count → all equal);
`WalkAttention` learns query·key weights (differ per source). Same support, different ability to select.

In [ ]:
g = 3
mask = torch.from_numpy((raw[g] > 0).T.astype('float32')).to_sparse().coalesce()
tgt, src = mask.indices()[0], mask.indices()[1]
fixed = raw[g][src.numpy(), tgt.numpy()].astype(float)
torch.manual_seed(0)
layer = WalkAttention(data.x.size(1), 8, n_heads=1)
q = layer.q(data.x).view(N,1,8); k = layer.k(data.x).view(N,1,8)
alpha = pyg_softmax((q[tgt]*k[src]).sum(-1)*layer.scale, tgt, num_nodes=N).detach().numpy().ravel()
into_t = (tgt.numpy() == t)
fixed_n = fixed[into_t] / fixed[into_t].sum()
viz.plot_fixed_vs_learned(src.numpy()[into_t], fixed_n, alpha[into_t],
                          'weights into the target (same sources)')

## Figure 4 — the learned attention as a matrix

The attention weights, nonzero only on walk-reachable pairs. Compare with Figure 2: same sparse footprint, but
now each entry is a *learned* number, not a fixed count.

**One sentence:** the algebra says WHO attends to whom; attention learns HOW MUCH. Proof in P5.

In [ ]:
Amat = np.zeros((N, N))
Amat[tgt.numpy(), src.numpy()] = alpha
viz.plot_multiplicity_heatmap(Amat, 'learned attention weights at range 4')